# Chapter 17 — Mixed Precision (BF16 / FP16)

> Course: **llm.c — Zero to Hero**, Chapter 17 of ~20.
> Builds on Chapters 9-16.

GPT-2 124M is **500 MB** of weights in FP32. In **bfloat16** it's 250 MB. Tensor cores run BF16 matmuls **2-4× faster** than FP32. Mixed precision is a free lunch — *if* you do it carefully.

This chapter is about how `llm.c` does it: the `floatX` typedef that lets one source file produce FP32 or BF16 binaries, the **master weights** that keep AdamW stable, and the casting/scaling tricks that prevent gradient underflow. We won't compile a full mixed-precision kernel here (that's `train_gpt2.cu`), but we'll do small BF16 demos to ground the concepts.

### Learning objectives

By the end of this chapter you will:

- Compare BF16, FP16, and FP32 ranges and precisions.
- Explain why **master FP32 weights** are required even with BF16 forward/backward.
- Read the `floatX` typedef in `llmc/cuda_common.h` and the `ENABLE_BF16` switch.
- Cast between FP32 and BF16 in CUDA via `__float2bfloat16` / `__bfloat162float`.


## 1. Concept — Three Floats Walk Into a Bar

| | FP32 | FP16 | BF16 |
|---|---|---|---|
| Bits | 32 (1+8+23) | 16 (1+5+10) | 16 (1+8+7) |
| Range | ±10³⁸ | ±10⁵ | ±10³⁸ |
| Precision (~ULP near 1.0) | 10⁻⁷ | 10⁻³ | 10⁻² |
| Tensor-core speedup | 1× | 2-4× | 2-4× |

Two things to internalize:

1. **BF16 has the same exponent as FP32 (8 bits) but only 7 mantissa bits.** Range is identical to FP32 — no overflow when small gradients turn into very small numbers. Precision is much worse (~3 decimal digits) but Transformer weights and activations don't *need* that precision once trained.
2. **FP16 has only 5 exponent bits.** Range tops out around `65504`. A gradient that's `1e-7 × something_big` can underflow to zero. To use FP16 you need **loss scaling**: multiply the loss by a constant before backward, then divide it out after. BF16 sidesteps the whole issue.

`llm.c` uses **BF16** by default for GPU training (`ENABLE_BF16`). It's the modern standard and what the H100 / A100 tensor cores prefer.


## 2. Concept — Why Master Weights Matter

Here's the catastrophic failure mode if you train with naive BF16:

```
For each step:
    grad = backward()                          # bf16
    momentum = 0.9 * momentum + 0.1 * grad     # bf16 update accumulates noise
    param = param - 1e-4 * momentum            # bf16 param has 7-bit precision
```

The `1e-4 * momentum` is a **tiny** number. When you add it to `param` (which is roughly `O(1)`), you're trying to add `~1e-5` to `~1.0`. In BF16 that's *below* the representable next number — the addition rounds to zero and the parameter doesn't update. After thousands of steps with no update, training stalls.

Fix: keep the **canonical parameter values in FP32** (master weights). Do the optimizer math in FP32. After each step, *cast* the master weights to BF16 to use them in forward/backward.

```
For each step:
    forward(bf16_params)                          # 2× faster
    grads_bf16 = backward()                       # 2× faster
    grads_fp32 = (float) grads_bf16
    momentum_fp32 = 0.9 * momentum_fp32 + 0.1 * grads_fp32
    master_fp32 -= 1e-4 * momentum_fp32           # full precision update
    bf16_params = (bf16) master_fp32              # cast for next forward
```

Memory cost: master weights double parameter storage (FP32 for master + BF16 for working). Speed cost: a single cast kernel per step. Stability gain: enormous.

In `train_gpt2.cu`, the master weights live in `params_memory` (FP32 always); `params_memory_pre_cast` (or similar in newer versions) holds the BF16 working copy. Each step casts after AdamW.


## 3. The `floatX` Typedef

`llm.c`'s clever trick: **one source file, two precisions, compile-time switch**. From `llmc/cuda_common.h`:

```cpp
#if defined(ENABLE_BF16)
typedef __nv_bfloat16 floatX;
typedef __nv_bfloat16 floatN;
#define CUBLAS_LOWP CUDA_R_16BF
// ...
#elif defined(ENABLE_FP16)
typedef half floatX;
typedef half floatN;
#define CUBLAS_LOWP CUDA_R_16F
// ...
#else
typedef float floatX;
typedef float floatN;
#define CUBLAS_LOWP CUDA_R_32F
// ...
#endif
```

Every layer in `llmc/*.cuh` is written against `floatX`:

```cpp
__global__ void layernorm_forward_kernel6(floatX* out, ..., const floatX* inp, ...);
```

With `ENABLE_BF16`, `floatX = __nv_bfloat16` and the kernel is BF16. Without it, `floatX = float` and the same source compiles to FP32. The build system flips `ENABLE_BF16` via the Makefile:

```
make train_gpt2cu                  # default: ENABLE_BF16=1, gives BF16 build
make train_gpt2fp32cu              # explicit: FP32 build
```

This is genuinely elegant — the entire 1900-line `train_gpt2.cu` doesn't have a single explicit BF16 type anywhere; it just uses `floatX` and lets `cuda_common.h` decide.

The `Packed128` we mentioned in Chapter 11 is also templated on `floatX` — its `size` is `16/sizeof(floatX)`, so 8 BF16s per 128-bit transaction or 4 FP32s.


## 4. Demo — BF16 Round-Trip

In [ ]:
!mkdir -p course/ch17_build


In [ ]:
%%writefile course/ch17_build/bf16_demo.cu
#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda_runtime.h>
#include <cuda_bf16.h>

// Cast FP32 → BF16 → FP32 elementwise, measure how much precision is lost
__global__ void roundtrip_bf16(float* out, const float* inp, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        __nv_bfloat16 v_bf16 = __float2bfloat16(inp[i]);
        out[i] = __bfloat162float(v_bf16);
    }
}

// Sum two FP32 numbers via BF16 intermediate to demo "small grad lost in big param"
__global__ void bad_update_bf16(__nv_bfloat16* params, const __nv_bfloat16* grads, float lr, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        // Naive bf16 update: param -= lr*grad, all in bf16 → precision loss
        float p = __bfloat162float(params[i]);
        float g = __bfloat162float(grads[i]);
        params[i] = __float2bfloat16(p - lr * g);
    }
}

// Master-weights update: FP32 master, BF16 working copy
__global__ void good_update_master(float* master, __nv_bfloat16* working,
                                   const __nv_bfloat16* grads, float lr, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        float g = __bfloat162float(grads[i]);
        master[i] -= lr * g;                                // full FP32 precision
        working[i] = __float2bfloat16(master[i]);            // cast for next forward
    }
}

int main(void) {
    int N = 8;
    float h_inp[] = {1.0f, 0.5f, 1.234567f, 1e-3f, 1.0f + 1e-5f, 1e10f, -123.456f, 0.001f + 1.0f};
    float h_rt[8];
    float* d_inp; float* d_out;
    cudaMalloc(&d_inp, 8*4); cudaMalloc(&d_out, 8*4);
    cudaMemcpy(d_inp, h_inp, 8*4, cudaMemcpyHostToDevice);
    roundtrip_bf16<<<1, 8>>>(d_out, d_inp, 8);
    cudaMemcpy(h_rt, d_out, 8*4, cudaMemcpyDeviceToHost);
    printf("BF16 round-trip precision:\n");
    printf("  fp32 in       bf16 out      diff\n");
    for (int i = 0; i < 8; i++) {
        printf("  %12.6e  %12.6e  %.2e\n", h_inp[i], h_rt[i], fabsf(h_inp[i] - h_rt[i]));
    }

    // Demo: 1000 steps of "param -= lr*grad" with lr*grad << param. Bf16 vs master.
    int M = 1;
    __nv_bfloat16* d_param_bf16; __nv_bfloat16* d_grads_bf16;
    float* d_master;             __nv_bfloat16* d_working;
    cudaMalloc(&d_param_bf16, 2*M); cudaMalloc(&d_grads_bf16, 2*M);
    cudaMalloc(&d_master,     4*M); cudaMalloc(&d_working,    2*M);

    float p0 = 1.0f, g0 = 0.001f, lr = 1e-4f;
    int steps = 1000;
    // After 1000 steps the *true* update should be 1.0 - 1000 * 1e-4 * 1e-3 = 1.0 - 1e-4 = 0.9999
    __nv_bfloat16 h_p0 = __float2bfloat16(p0); __nv_bfloat16 h_g0 = __float2bfloat16(g0);
    cudaMemcpy(d_param_bf16, &h_p0, 2, cudaMemcpyHostToDevice);
    cudaMemcpy(d_grads_bf16, &h_g0, 2, cudaMemcpyHostToDevice);
    cudaMemcpy(d_master,     &p0,   4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_working,    &h_p0, 2, cudaMemcpyHostToDevice);

    for (int s = 0; s < steps; s++) {
        bad_update_bf16<<<1,1>>>(d_param_bf16, d_grads_bf16, lr, M);
        good_update_master<<<1,1>>>(d_master, d_working, d_grads_bf16, lr, M);
    }
    __nv_bfloat16 h_naive; float h_master_val;
    cudaMemcpy(&h_naive,      d_param_bf16, 2, cudaMemcpyDeviceToHost);
    cudaMemcpy(&h_master_val, d_master,     4, cudaMemcpyDeviceToHost);
    printf("\nAfter 1000 steps of param -= 1e-4 * 1e-3 (true result: 0.9999):\n");
    printf("  naive bf16 param    : %.6f   <-- often fails to update\n", __bfloat162float(h_naive));
    printf("  fp32 master weights : %.6f   <-- correct\n", h_master_val);

    cudaFree(d_inp); cudaFree(d_out);
    cudaFree(d_param_bf16); cudaFree(d_grads_bf16); cudaFree(d_master); cudaFree(d_working);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch17_build/bf16_demo course/ch17_build/bf16_demo.cu && ./course/ch17_build/bf16_demo


Look at the `1.0 + 1e-5f` row: BF16 silently rounds it back to `1.0` — the addition is below BF16's precision near 1.0. Now look at the **after 1000 steps** result: the naive BF16 parameter typically **fails to move at all** (stays at 1.0), while the FP32 master correctly arrives near `0.9999`. **This is exactly the failure mode that motivates master weights.**


## 5. The `cast_kernel` Pattern

After every AdamW step, `train_gpt2.cu` casts the FP32 master weights to BF16:

```cpp
__global__ void copy_and_cast_kernel(floatX* dst, const float* src, size_t n) {
    size_t i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) dst[i] = (floatX) src[i];
}
```

(The actual production kernel uses `Packed128` for vectorized stores, but the structure is this.) This kernel runs once per training step over the full 124M parameters — it's bandwidth-bound and takes microseconds.

The complete training step sequence with mixed precision:

```
1. forward (BF16):      reads bf16 params, writes bf16 activations
2. backward (BF16):     reads bf16 activations and params, writes bf16 grads
3. AdamW (FP32):        m, v, master_params all FP32; cast bf16 grads in
4. cast (FP32 -> BF16): copy master params -> bf16 working copy for next forward
```

Steps 1-2 run at ~2× the FP32 speed. Step 3 stays in FP32 for stability. Step 4 is cheap. Net: ~2× speedup on the dominant cost (forward+backward) at no stability cost.


## 6. Translation Bridge

| Concept | PyTorch | `llm.c` |
|---|---|---|
| Mixed-precision training | `torch.autocast(dtype=torch.bfloat16)` | `floatX` typedef + master weights |
| Master weights | `optimizer.scale.master_weights = True` | always-on FP32 `params_memory` |
| Loss scaling (FP16 only) | `torch.amp.GradScaler` | `ENABLE_FP16` build with explicit scaling code |
| Tensor-core matmul | Done internally | `cublasLtMatmul` with `CUBLAS_LOWP` data type |


## 7. TODO Exercise — Master-Weights Update

In [ ]:
%%writefile course/ch17_build/exercise1.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cuda_bf16.h>

// TODO: Implement the master-weights update.
// Inputs: master (fp32), bf16_grads, lr.
// Outputs: master updated, working_bf16 = (bf16) master.
__global__ void master_update(float* master, __nv_bfloat16* working,
                              const __nv_bfloat16* grads, float lr, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        // TODO: convert grads[i] from bf16 to fp32 with __bfloat162float
        // TODO: master[i] -= lr * g
        // TODO: working[i] = __float2bfloat16(master[i])
    }
}

int main(void) {
    int N = 4;
    float h_master[] = {1.0f, 2.0f, 3.0f, 4.0f};
    __nv_bfloat16 h_grads[4]; for (int i = 0; i < 4; i++) h_grads[i] = __float2bfloat16(0.5f * (i+1));
    __nv_bfloat16 h_working[4]; for (int i = 0; i < 4; i++) h_working[i] = __float2bfloat16(h_master[i]);

    float* d_master; __nv_bfloat16 *d_grads, *d_working;
    cudaMalloc(&d_master, N*4); cudaMalloc(&d_grads, N*2); cudaMalloc(&d_working, N*2);
    cudaMemcpy(d_master, h_master, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_grads, h_grads, N*2, cudaMemcpyHostToDevice);
    cudaMemcpy(d_working, h_working, N*2, cudaMemcpyHostToDevice);

    master_update<<<1, N>>>(d_master, d_working, d_grads, 0.1f, N);
    cudaMemcpy(h_master, d_master, N*4, cudaMemcpyDeviceToHost);
    cudaMemcpy(h_working, d_working, N*2, cudaMemcpyDeviceToHost);

    // Expected after step:
    // i=0: master = 1.0 - 0.1*0.5 = 0.95   working ≈ 0.95
    // i=1: 2.0 - 0.1*1.0 = 1.9              ≈ 1.9
    // i=2: 3.0 - 0.1*1.5 = 2.85             ≈ 2.85
    // i=3: 4.0 - 0.1*2.0 = 3.8              ≈ 3.8
    int ok = 1;
    float expect[] = {0.95f, 1.9f, 2.85f, 3.8f};
    for (int i = 0; i < N; i++) {
        if (fabsf(h_master[i] - expect[i]) > 1e-3) ok = 0;
        printf("master[%d] = %.4f  (expect %.4f)   working = %.4f\n",
               i, h_master[i], expect[i], __bfloat162float(h_working[i]));
    }
    printf("%s\n", ok ? "PASS" : "FAIL");
    cudaFree(d_master); cudaFree(d_grads); cudaFree(d_working);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch17_build/exercise1 course/ch17_build/exercise1.cu && ./course/ch17_build/exercise1


### Solution

In [ ]:
%%writefile course/ch17_build/exercise1_sol.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <cuda_bf16.h>

__global__ void master_update(float* master, __nv_bfloat16* working,
                              const __nv_bfloat16* grads, float lr, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        float g = __bfloat162float(grads[i]);
        master[i] -= lr * g;
        working[i] = __float2bfloat16(master[i]);
    }
}

int main(void) {
    int N = 4;
    float h_master[] = {1.0f, 2.0f, 3.0f, 4.0f};
    __nv_bfloat16 h_grads[4]; for (int i = 0; i < 4; i++) h_grads[i] = __float2bfloat16(0.5f * (i+1));
    __nv_bfloat16 h_working[4]; for (int i = 0; i < 4; i++) h_working[i] = __float2bfloat16(h_master[i]);
    float* d_master; __nv_bfloat16 *d_grads, *d_working;
    cudaMalloc(&d_master, N*4); cudaMalloc(&d_grads, N*2); cudaMalloc(&d_working, N*2);
    cudaMemcpy(d_master, h_master, N*4, cudaMemcpyHostToDevice);
    cudaMemcpy(d_grads, h_grads, N*2, cudaMemcpyHostToDevice);
    cudaMemcpy(d_working, h_working, N*2, cudaMemcpyHostToDevice);
    master_update<<<1, N>>>(d_master, d_working, d_grads, 0.1f, N);
    cudaMemcpy(h_master, d_master, N*4, cudaMemcpyDeviceToHost);
    cudaMemcpy(h_working, d_working, N*2, cudaMemcpyDeviceToHost);
    int ok = 1;
    float expect[] = {0.95f, 1.9f, 2.85f, 3.8f};
    for (int i = 0; i < N; i++) {
        if (fabsf(h_master[i] - expect[i]) > 1e-3) ok = 0;
        printf("master[%d] = %.4f  (expect %.4f)   working = %.4f\n",
               i, h_master[i], expect[i], __bfloat162float(h_working[i]));
    }
    printf("%s\n", ok ? "PASS" : "FAIL");
    cudaFree(d_master); cudaFree(d_grads); cudaFree(d_working);
    return 0;
}


In [ ]:
!nvcc -O2 -o course/ch17_build/exercise1_sol course/ch17_build/exercise1_sol.cu && ./course/ch17_build/exercise1_sol


## Recap

You now know:

- BF16 has FP32's exponent range with FP16's storage size — modern standard for ML.
- Naive BF16 training fails because tiny optimizer updates round to zero. **Master FP32 weights** are required.
- `floatX` is a templated typedef that lets one source file produce FP32 or BF16 binaries via the `ENABLE_BF16` flag.
- Mixed-precision training: BF16 forward+backward (~2× speedup), FP32 AdamW + master weights for stability, cast back to BF16 each step.

### What's next

**Chapter 18 — GPU Optimizer & Gradient Clipping.** AdamW you've already written; now we'll see it on GPU as `adamw_kernel3` in `llmc/adamw.cuh`, plus the **global gradient norm** computation in `llmc/global_norm.cuh` (a *two-stage* reduction over all 124M parameters).

When you're ready, say **"proceed to Chapter 18"**.
